Installing libraries

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 52.1 MB/s eta 0:00:00


In [ ]:
!pip install -q -U "peft>=0.19.0"

Imports, loading the drive, and student model

In [ ]:
import torch, pandas as pd, json, re, glob, os, time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
from huggingface_hub import login
from google.colab import userdata, drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Research Project Synthetic Data'
login(userdata.get('HF_TOKEN'))

MODEL_ID = "google/gemma-4-E4B-it"
os.makedirs(f'{BASE}/adapters', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Evaluation prompt

In [ ]:
def build_eval_prompt(row):
    """Prompts the model to answer questions and respond with a single letter"""
    return (f"Answer this medical multiple-choice question with a single letter.\n\n"
            f"Question: {row['question']}\n"
            f"A) {row['opa']}\nB) {row['opb']}\nC) {row['opc']}\nD) {row['opd']}\n\n"
            f"Answer:")

Loading the synthetic data

In [ ]:
pool = pd.concat([pd.read_json(f, lines=True) for f in
                  sorted(glob.glob(f'{BASE}/generated/*.json'))], ignore_index=True)
print(f"{len(pool)} questions across {pool['subject_name'].nunique()} subjects")

3200 questions across 16 subjects


In [ ]:
def make_training_text(row, tokenizer):
    """Full conversation: prompt + correct answer, as the model should learn it."""
    msgs = [
        {"role": "user", "content": build_eval_prompt(row)},
        {"role": "assistant", "content": 'ABCD'[int(row['cop']) - 1]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

def build_dataset(df, tokenizer):
    texts = [make_training_text(r, tokenizer) for _, r in df.iterrows()]
    return Dataset.from_dict({"text": texts})

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"


def fresh_model():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map="auto", torch_dtype=torch.bfloat16)
    lora = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=r".*language_model\..*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)",
    )
    m = get_peft_model(m, lora)
    m.print_trainable_parameters()
    return m

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

In [ ]:
def train_condition(df, run_name, epochs=3, lr=5e-4):
    model = fresh_model()
    ds = build_dataset(df, tokenizer)

    cfg = SFTConfig(
        output_dir=f'/content/out_{run_name}',
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=lr,
        logging_steps=10,
        save_strategy="no",
        bf16=True,
        max_length=1024,              # was max_seq_length
        report_to="none",
        # dataset_text_field defaults to "text" — matches your dataset
    )

    trainer = SFTTrainer(model=model, train_dataset=ds, args=cfg)

    t0 = time.time()
    trainer.train()
    mins = (time.time() - t0) / 60

    path = f'{BASE}/adapters/{run_name}'
    trainer.model.save_pretrained(path)
    print(f"\n{run_name}: trained on {len(df)} examples in {mins:.1f} min → {path}")
    return path, mins

Loading all subjects

In [ ]:
with open(f'{BASE}/Taxonomy/taxonomy_filtered.json') as f:
    taxonomy = json.load(f)
subjects = list(taxonomy.keys())
print(f"{len(subjects)} subjects")


16 subjects


defining conditions

In [ ]:
import random, json, os, gc, torch

def get_subjects_for(coverage_pct, seed):
    n = round(len(subjects) * coverage_pct / 100)
    rng = random.Random(seed)
    return sorted(rng.sample(subjects, n))

CONDITIONS = {}
for seed in [1, 2]:
    for pct in [100, 75, 50, 25]:
        name = f"cov{pct}_seed{seed}"
        CONDITIONS[name] = get_subjects_for(pct, seed)

for name, subs in CONDITIONS.items():
    done = "✓" if os.path.exists(f'{BASE}/adapters/{name}') else " "
    print(f"[{done}] {name:16} {len(subs):2} subjects")

[✓] cov100_seed1     16 subjects
[✓] cov75_seed1      12 subjects
[✓] cov50_seed1       8 subjects
[✓] cov25_seed1       4 subjects
[✓] cov100_seed2     16 subjects
[✓] cov75_seed2      12 subjects
[✓] cov50_seed2       8 subjects
[✓] cov25_seed2       4 subjects


training on a test slice of 400

In [ ]:
test_slice = pool.sample(400, random_state=1)
train_condition(test_slice, run_name="_sanity", epochs=1, lr=5e-4)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.400616
20,0.937417



_sanity: trained on 400 examples in 2.2 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/_sanity


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/_sanity',
 2.17041464249293)

Extra seeds at unstable points (Was added after evaluating two seeds for each coverage codition)

In [ ]:
EXTRA_SEEDS = {
    25: [3, 4, 5],
    50: [3, 4],
}

EXTRA = {}
for pct, seeds in EXTRA_SEEDS.items():
    for s in seeds:
        name = f"cov{pct}_seed{s}"
        EXTRA[name] = get_subjects_for(pct, s)

for name, subs in EXTRA.items():
    done = "✓" if os.path.exists(f'{BASE}/adapters/{name}') else " "
    print(f"[{done}] {name:16} {len(subs):2} subjects: {subs}")

[✓] cov25_seed3       4 subjects: ['dental', 'microbiology', 'ophthalmology', 'pathology']
[✓] cov25_seed4       4 subjects: ['biochemistry', 'forensic medicine', 'microbiology', 'pharmacology']
[✓] cov25_seed5       4 subjects: ['gynaecology & obstetrics', 'ophthalmology', 'pharmacology', 'physiology']
[✓] cov50_seed3       8 subjects: ['biochemistry', 'dental', 'gynaecology & obstetrics', 'microbiology', 'ophthalmology', 'pathology', 'social & preventive medicine', 'surgery']
[ ] cov50_seed4       8 subjects: ['biochemistry', 'dental', 'forensic medicine', 'medicine', 'microbiology', 'pharmacology', 'radiology', 'surgery']


Reproducing the first Invalid run (was earlier removed from the notebook)

In [ ]:
def fresh_model_v1():
    """ORIGINAL configuration — PEFT defaults (q_proj/v_proj only).
    Documented for Section 4.4: produced an adapter too small to affect inference."""
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config,
        device_map="auto", torch_dtype=torch.bfloat16)
    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM",
        # no target_modules — PEFT defaults adapt only q_proj and v_proj
    )
    m = get_peft_model(m, lora)
    m.print_trainable_parameters()
    return m

In [ ]:
_original = fresh_model
fresh_model = fresh_model_v1

train_condition(pool, run_name="_diagnostic_underpowered", epochs=3, lr=2e-4)

fresh_model = _original      # restores immediately

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

...

Truncating train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,2.592567
20,1.593818
30,1.233340
40,1.088713
50,1.024200
60,1.012852
70,0.982948
80,0.965138
90,0.964909
100,0.961112



_diagnostic_underpowered: trained on 3200 examples in 42.6 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/_diagnostic_underpowered


inspection of the invalidated first run

In [ ]:
from safetensors.torch import load_file

def inspect_adapter(name):
    w = load_file(f'{BASE}/adapters/{name}/adapter_model.safetensors')
    b_keys = [k for k in w if 'lora_B' in k]
    a_keys = [k for k in w if 'lora_A' in k]
    print(f"{name}")
    print(f"  lora_B modules: {len(b_keys)}")
    print(f"  lora_B max magnitude: {max(w[k].abs().max().item() for k in b_keys):.6f}")
    print(f"  lora_A max magnitude: {max(w[k].abs().max().item() for k in a_keys):.6f}")

inspect_adapter("_diagnostic_underpowered")
inspect_adapter("cov100_seed1")          # the working config, for comparison

_diagnostic_underpowered
  lora_B modules: 66
  lora_B max magnitude: 0.024170
  lora_A max magnitude: 0.033447
cov100_seed1
  lora_B modules: 258
  lora_B max magnitude: 0.041504
  lora_A max magnitude: 0.062500


100% coverage

In [ ]:

train_condition(pool, run_name="cov100_seed1")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

trainable params: 34,881,536 || all params: 7,975,982,368 || trainable%: 0.4373


Adding EOS to train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.375309
20,0.960335
30,0.934913
40,0.942712
50,0.905323
60,0.890149
70,0.883375
80,0.868109
90,0.856131
100,0.871178



cov100_seed1: trained on 3200 examples in 51.4 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_seed1


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_seed1',
 51.3826602379481)

In [ ]:
RUN = "cov100_seed2"

subs = CONDITIONS[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects")
print(f"subjects: {subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)

cov100_seed2: 3200 examples, 16 subjects
subjects: ['anatomy', 'biochemistry', 'dental', 'ent', 'forensic medicine', 'gynaecology & obstetrics', 'medicine', 'microbiology', 'ophthalmology', 'pathology', 'pediatrics', 'pharmacology', 'physiology', 'radiology', 'social & preventive medicine', 'surgery']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.379504
20,0.961151
30,0.934908
40,0.943251
50,0.905973
60,0.889860
70,0.884304
80,0.868321
90,0.856895
100,0.870601



cov100_seed2: trained on 3200 examples in 51.5 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_seed2


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_seed2',
 51.469972813129424)

100% coverage with different training parameters

In [ ]:
train_condition(pool, run_name="cov100_r32", epochs=4, lr=5e-4)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

trainable params: 69,763,072 || all params: 8,010,863,904 || trainable%: 0.8709


Adding EOS to train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/3200 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.336749
20,0.966934
30,0.942859
40,0.953712
50,0.916285
60,0.902526
70,0.897069
80,0.886771
90,0.877000
100,0.895515



cov100_r32: trained on 3200 examples in 70.6 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_r32


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_r32',
 70.59964516560237)

25% coverage

In [ ]:
import random
random.seed(1)
subs_25 = random.sample(subjects, 4)
print("25% subjects:", subs_25)

df_25 = pool[pool['subject_name'].isin(subs_25)]
print(f"{len(df_25)} training examples")

train_condition(df_25, run_name="cov25_seed1")

25% subjects: ['forensic medicine', 'pathology', 'radiology', 'physiology']
800 training examples


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.433821
20,0.961605
30,0.899284
40,0.918289
50,0.833253
60,0.602848
70,0.565089
80,0.579244
90,0.571467
100,0.555234



cov25_seed1: trained on 800 examples in 13.1 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov25_seed1


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov25_seed1',
 13.100478184223174)

seed 2

In [ ]:
RUN = "cov25_seed2"

subs = CONDITIONS[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects")
print(f"subjects: {subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)

cov25_seed2: 800 examples, 4 subjects
subjects: ['biochemistry', 'gynaecology & obstetrics', 'social & preventive medicine', 'surgery']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.321369
20,0.899005
30,0.836203
40,0.803497
50,0.791682
60,0.549226
70,0.555332
80,0.518489
90,0.509783
100,0.495221



cov25_seed2: trained on 800 examples in 12.6 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov25_seed2


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov25_seed2',
 12.643170106410981)

In [ ]:
RUN = "cov25_seed3"

subs = EXTRA[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects\n{subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)
gc.collect(); torch.cuda.empty_cache()

cov25_seed3: 800 examples, 4 subjects
['dental', 'microbiology', 'ophthalmology', 'pathology']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.473221
20,0.944307
30,0.894257
40,0.852571
50,0.886203
60,0.607683
70,0.596747
80,0.570346
90,0.578514
100,0.572140



cov25_seed3: trained on 800 examples in 12.8 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov25_seed3


In [ ]:
RUN = "cov25_seed4"

subs = EXTRA[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects\n{subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)
gc.collect(); torch.cuda.empty_cache()

cov25_seed4: 800 examples, 4 subjects
['biochemistry', 'forensic medicine', 'microbiology', 'pharmacology']



Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

trainable params: 34,881,536 || all params: 7,975,982,368 || trainable%: 0.4373


Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

...

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.344319
20,0.918162
30,0.881717
40,0.811687
50,0.834426
60,0.534726
70,0.524467
80,0.535301
90,0.487415
100,0.499428



cov25_seed4: trained on 800 examples in 12.5 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov25_seed4


In [ ]:
RUN = "cov25_seed5"

subs = EXTRA[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects\n{subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)
gc.collect(); torch.cuda.empty_cache()

cov25_seed5: 800 examples, 4 subjects
['gynaecology & obstetrics', 'ophthalmology', 'pharmacology', 'physiology']



Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

trainable params: 34,881,536 || all params: 7,975,982,368 || trainable%: 0.4373


Adding EOS to train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

...

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.299640
20,0.849852
30,0.780112
40,0.805848
50,0.743493
60,0.546110
70,0.513342
80,0.527955
90,0.487488
100,0.469652



cov25_seed5: trained on 800 examples in 12.6 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov25_seed5


Half Data run, all 16 subjects half data.

In [ ]:
# stratified half, 100 per subject rather than 200, keeping coverage constant
half_pool = (pool.groupby('subject_name', group_keys=False)
                 .apply(lambda g: g.sample(frac=0.5, random_state=42)))

print(f"{len(half_pool)} examples across {half_pool['subject_name'].nunique()} subjects")
print(half_pool['subject_name'].value_counts().head())

1600 examples across 16 subjects
subject_name
anatomy              100
biochemistry         100
dental               100
ent                  100
forensic medicine    100
Name: count, dtype: int64


/tmp/ipykernel_1431/2751370724.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=0.5, random_state=42)))


In [ ]:
train_condition(half_pool, run_name="cov100_half", epochs=3, lr=5e-4)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.419547
20,0.960507
30,0.954046
40,0.917879
50,0.879595
60,0.876943
70,0.867119
80,0.850424
90,0.887047
100,0.858080



cov100_half: trained on 1600 examples in 24.9 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_half


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov100_half',
 24.94936753908793)

50% coverage training

In [ ]:
RUN = "cov50_seed1"

subs = CONDITIONS[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects")
print(f"subjects: {subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)

cov50_seed1: 1600 examples, 8 subjects
subjects: ['biochemistry', 'forensic medicine', 'microbiology', 'pathology', 'pharmacology', 'physiology', 'radiology', 'surgery']



Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

trainable params: 34,881,536 || all params: 7,975,982,368 || trainable%: 0.4373


Adding EOS to train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.315269
20,0.933323
30,0.915780
40,0.874006
50,0.894387
60,0.852100
70,0.784402
80,0.812212
90,0.801192
100,0.819685



cov50_seed1: trained on 1600 examples in 25.9 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov50_seed1


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov50_seed1',
 25.865677905082702)

In [ ]:
RUN = "cov50_seed2"

subs = CONDITIONS[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects")
print(f"subjects: {subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)

cov50_seed2: 1600 examples, 8 subjects
subjects: ['biochemistry', 'dental', 'forensic medicine', 'gynaecology & obstetrics', 'pathology', 'pediatrics', 'social & preventive medicine', 'surgery']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.442558
20,0.980787
30,0.957916
40,0.922233
50,0.896256
60,0.870512
70,0.829145
80,0.863040
90,0.822174
100,0.883243



cov50_seed2: trained on 1600 examples in 26.1 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov50_seed2


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov50_seed2',
 26.0694757938385)

In [ ]:
RUN = "cov50_seed3"

subs = EXTRA[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects\n{subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)
gc.collect(); torch.cuda.empty_cache()

cov50_seed3: 1600 examples, 8 subjects
['biochemistry', 'dental', 'gynaecology & obstetrics', 'microbiology', 'ophthalmology', 'pathology', 'social & preventive medicine', 'surgery']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Truncating train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.422632
20,0.960653
30,0.926338
40,0.896223
50,0.882375
60,0.839300
70,0.860960
80,0.857138
90,0.778913
100,0.825907



cov50_seed3: trained on 1600 examples in 25.8 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov50_seed3


In [ ]:
RUN = "cov50_seed4"

subs = EXTRA[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects\n{subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)
gc.collect(); torch.cuda.empty_cache()

cov50_seed4: 1600 examples, 8 subjects
['biochemistry', 'dental', 'forensic medicine', 'medicine', 'microbiology', 'pharmacology', 'radiology', 'surgery']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Truncating train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1600 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.369927
20,0.953555
30,0.964524
40,0.922198
50,0.862443
60,0.896430
70,0.807388
80,0.842919
90,0.830871
100,0.841674



cov50_seed4: trained on 1600 examples in 25.6 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov50_seed4


75% coverage training

In [ ]:
RUN = "cov75_seed1"

subs = CONDITIONS[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects")
print(f"subjects: {subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)

gc.collect(); torch.cuda.empty_cache()

cov75_seed1: 2400 examples, 12 subjects
subjects: ['biochemistry', 'ent', 'forensic medicine', 'gynaecology & obstetrics', 'medicine', 'microbiology', 'ophthalmology', 'pathology', 'pharmacology', 'physiology', 'radiology', 'surgery']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Truncating train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.395772
20,0.894990
30,0.892073
40,0.904986
50,0.888926
60,0.909538
70,0.902345
80,0.861546
90,0.861056
100,0.858759



cov75_seed1: trained on 2400 examples in 38.4 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov75_seed1


In [ ]:
RUN = "cov75_seed2"

subs = CONDITIONS[RUN]
df = pool[pool['subject_name'].isin(subs)]
print(f"{RUN}: {len(df)} examples, {len(subs)} subjects")
print(f"subjects: {subs}\n")

train_condition(df, run_name=RUN, epochs=3, lr=5e-4)

cov75_seed2: 2400 examples, 12 subjects
subjects: ['anatomy', 'biochemistry', 'dental', 'ent', 'forensic medicine', 'gynaecology & obstetrics', 'medicine', 'ophthalmology', 'pathology', 'pediatrics', 'social & preventive medicine', 'surgery']



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

...

Dropping fully masked examples from train dataset:   0%|          | 0/2400 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss
10,1.370838
20,0.974098
30,0.952747
40,0.932658
50,0.858341
60,0.909250
70,0.886462
80,0.877419
90,0.882568
100,0.846307



cov75_seed2: trained on 2400 examples in 38.9 min → /content/drive/MyDrive/Research Project Synthetic Data/adapters/cov75_seed2


('/content/drive/MyDrive/Research Project Synthetic Data/adapters/cov75_seed2',
 38.94304163853327)